<div style="border-radius: 10px; padding: 32px 0px; border: 1px solid rgba(128,128,128,0.2);">
  <div style="display: flex; justify-content: space-between; align-items: flex-start; flex-wrap: wrap; gap: 16px; padding: 0px 32px;">
  <div>
    <div style="font-size: 0.75rem; letter-spacing: 3px; text-transform: uppercase; font-weight: 600; margin-bottom: 10px; opacity: 0.6;">
      Máster Universitario en Big Data y Computación en la Nube.
    </div>
    <div style="font-size: 1.5rem; font-weight: 700; margin-bottom: 4px;">Trabajo de Fin de Máster</div>
    <div style="font-size: 1rem; font-weight: 400; opacity: 0.75;">Clasificador taxonómico de boletines oficiales españoles</div>
  </div>
  <div style="margin-top: 16px; display: flex; align-items: center; gap: 12px;">
    <div style="font-size: 1rem; font-weight: 600;">Hugo de Lamo</div>
  </div>
  </div>
</div>

# 05 · Clasificador taxonómico de boletines oficiales

Este notebook implementa un clasificador multietiqueta de publicaciones de boletines oficiales españoles usando **Pydantic AI**. El problema es una aguja en un pajar: de ~65 000 publicaciones del Q1 2025, solo ~3,7 % son relevantes para el dominio ambiental-energético.

El clasificador responde cuatro preguntas por publicación:
1. **¿Es relevante?** — ¿Pertenece al universo de autorizaciones ambiental-energéticas?
2. **¿Qué procedimientos contiene?** — Lista multilabel: DIA, AAP, AAC, AAU, IIA, AAI, IAE, DUP.
3. **¿Qué tipo de acto es?** — Forma jurídica del documento (N1): resolución, anuncio, decreto…
4. **¿Qué tecnología menciona?** — Lista multilabel: fotovoltaica, eólica, hidrógeno…

---

## Estructura del notebook

### Parte I — Fundamentos
| § | Sección | Contenido |
|---|---------|----------|
| **0** | **Setup** | Entorno, dependencias, modelo local |
| **1** | **Schema de output** | `ClassifierOutput`, enums e invariantes |
| **2** | **Pre-procesamiento** | N0 lookup + N1 clasificador por reglas |
| **3** | **Ground truth** | Muestreo estratificado + anotación manual |
| **4** | **Agente base** | System prompt, construcción y casos cualitativos |

### Parte II — Ciclo de experimentación
| § | Sección | Contenido |
|---|---------|----------|
| **5** | **Experimento 1 — Baseline** | Zero-shot, sin contexto N1 |
| **6** | **Análisis de errores** | Qué falla y por qué |
| **7** | **Prompt v2** | Mejora basada en errores (DEC-013 a DEC-019) |
| **8** | **Experimento 2 — Prompt v2** | ¿Mejora respecto a baseline? |
| **9** | **Experimento 3 — Ablación +N1** | ¿Aporta el contexto de forma? |
| **10** | **Experimento 4 — Few-shot** | ¿Ayudan los ejemplos reales? |
| **11** | **Comparativa de modelos** | Qwen 3.5 9B vs Gemma 4 4B |

### Parte III — Análisis final
| § | Sección | Contenido |
|---|---------|----------|
| **12** | **Tabla resumen** | Comparativa de todos los experimentos |
| **13** | **Calibración de confianza** | ¿El modelo sabe cuándo no sabe? |
| **14** | **Conclusiones y trabajo futuro** | Hallazgos, limitaciones, v2 |

---

## §0. Setup

Cargamos las variables de entorno e importamos las librerías. El modelo vive en LM Studio — el único punto de cambio para conectar otro proveedor es `LM_STUDIO_MODEL`.

In [ ]:
import os
import html
import re
import json
import asyncio
from pathlib import Path

import pandas as pd
from dotenv import find_dotenv, load_dotenv
from pydantic_ai import Agent

load_dotenv(find_dotenv())

In [ ]:
from pydantic_ai.models.openai import OpenAIModel
from pydantic_ai.providers.openai import OpenAIProvider

# ── Modelo local via LM Studio ────────────────────────────────────────────────
# Cambiar LM_STUDIO_MODEL según el modelo cargado en LM Studio
LM_STUDIO_MODEL = "qwen/qwen3.5-9b"

model = OpenAIModel(
    LM_STUDIO_MODEL,
    provider=OpenAIProvider(
        base_url="http://localhost:1234/v1",
        api_key="lm-studio",
    ),
)

print(f"Modelo: {LM_STUDIO_MODEL} via LM Studio (localhost:1234)")

In [ ]:
import httpx

try:
    r = httpx.get("http://localhost:1234/v1/models", timeout=3)
    modelos = [m["id"] for m in r.json()["data"]]
    print(f"LM Studio operativo.")
    print(f"Modelos disponibles: {modelos}")
    assert LM_STUDIO_MODEL in modelos, f"{LM_STUDIO_MODEL} no está cargado"
    print(f"✓ {LM_STUDIO_MODEL} listo")
except httpx.ConnectError:
    raise RuntimeError("LM Studio no responde — ¿está el servidor arrancado?")

---

## §1. Schema de output

El schema define el **contrato entre el LLM y el sistema**: qué campos devuelve el modelo, de qué tipo y bajo qué restricciones. Pydantic valida cada respuesta antes de que llegue al resto del código, forzando un retry automático si algo no cumple el schema.

`ClassifierOutput` tiene 6 campos:

| Campo | Tipo | Rol |
|-------|------|-----|
| `is_relevant` | `bool` | ¿Pertenece al dominio ambiental-energético? |
| `act_type` | `ActType` | Forma jurídica del acto (N1) — resolución, anuncio, decreto… |
| `procedures` | `list[ProcedureType]` | Procedimientos identificados (N2) — multilabel |
| `technologies` | `list[TechnologyType]` | Tecnologías mencionadas (N3) — multilabel, puede ser vacía |
| `confidence` | `float` | Confianza global entre 0.0 y 1.0 |
| `reasoning` | `str` | Justificación breve citando el texto que dispara cada etiqueta |

Un `@model_validator` impone los invariantes de negocio: `is_relevant=True` exige `procedures != []`; `is_relevant=False` exige ambas listas vacías.

In [ ]:
from clasificador.schema import ActType, ProcedureType, TechnologyType, ClassifierOutput

In [ ]:
# Verificación de invariantes
ejemplo_valido = ClassifierOutput(
    is_relevant=True,
    act_type=ActType.RESOLUCION,
    procedures=[ProcedureType.DIA],
    technologies=[TechnologyType.FOTOVOLTAICA],
    confidence=0.98,
    reasoning="'se formula la declaración de impacto ambiental' → DIA. 'Planta Solar Fotovoltaica' → fotovoltaica.",
)
print("Ejemplo válido:")
print(ejemplo_valido.model_dump_json(indent=2))

print("\nViolación de invariante:")
try:
    ClassifierOutput(
        is_relevant=False, act_type=ActType.RESOLUCION,
        procedures=[ProcedureType.AAP], technologies=[],
        confidence=0.5, reasoning="Prueba.",
    )
except Exception as e:
    print(f"  ValidationError → {e.errors()[0]['msg']}")

---

## §2. Pre-procesamiento

Antes de llamar al LLM, cada registro pasa por dos pasos deterministas:

- **N0 — Ámbito**: lookup directo del campo `bulletin` → `estatal / autonómico / local`. Sin LLM.
- **N1 — Tipo de acto**: clasificador de primer token con pre-procesamiento de formatos especiales (BOCM, BOCA, BOE topónimos).

**Cobertura real medida**: 88.6% del corpus. El 11.4% restante cae en `OTROS` — principalmente topónimos BOE irrecuperables sin PDF.

In [ ]:
PATH_PARQUET = "../data/raw/silver_official_gazettes_2025_Q1.parquet"

df = pd.read_parquet(PATH_PARQUET)
df["description"] = df["description"].apply(html.unescape)

print(f"Corpus: {len(df):,} registros · {df['bulletin'].nunique()} boletines")

In [ ]:
# ── N0: ámbito por bulletin ───────────────────────────────────────────────────
_GAZETTE_TO_AMBITO = {
    "boe": "estatal",
    "madridambiental": "local",
    # resto → autonómico por defecto
}

def get_ambito(bulletin: str) -> str:
    return _GAZETTE_TO_AMBITO.get(bulletin.lower(), "autonómico")


# ── N1: tipo de acto por primer token ────────────────────────────────────────
def preprocess_description(desc: str, bulletin: str) -> str:
    desc = desc.strip()
    if "\n–" in desc:
        desc = desc.split("\n–", 1)[1].strip()
    elif "\n-" in desc:
        desc = desc.split("\n-", 1)[1].strip()
    if bulletin.lower() == "boca" and ".-" in desc:
        desc = desc.split(".-", 1)[1].strip()
    if re.match(r"^[A-ZÁÉÍÓÚÜÑ/\s]+$", desc) and len(desc.split()) <= 4:
        return "__TOPONIMO__"
    if desc.upper().startswith(("U.R.", "E.R.", "SUMA GESTIÓN", "ORGANISMO AUTÓNOMO DE HACIENDA")):
        return "__SUBASTA_AEAT__"
    return desc


_N1_MAP = [
    (r"corrección de errat",     ActType.CORRECCION_ERRORES),
    (r"corrección de error",     ActType.CORRECCION_ERRORES),
    (r"rectificación",           ActType.CORRECCION_ERRORES),
    (r"real decreto",            ActType.REAL_DECRETO),
    (r"orden foral",             ActType.ORDEN),
    (r"información pública",     ActType.INFORMACION_PUBLICA),
    (r"exposición pública",      ActType.INFORMACION_PUBLICA),
    (r"trámite de información",  ActType.INFORMACION_PUBLICA),
    (r"resolución",              ActType.RESOLUCION),
    (r"anuncio",                 ActType.ANUNCIO),
    (r"orden",                   ActType.ORDEN),
    (r"decreto foral",           ActType.DECRETO),
    (r"decreto",                 ActType.DECRETO),
    (r"acuerdo",                 ActType.ACUERDO),
    (r"aprobación",              ActType.APROBACION),
    (r"extracto",                ActType.EXTRACTO),
    (r"convenio",                ActType.CONVENIO),
    (r"adenda",                  ActType.CONVENIO),
    (r"solicitud",               ActType.SOLICITUD),
    (r"modificación",            ActType.MODIFICACION),
    (r"edicto",                  ActType.EDICTO),
    (r"notificación",            ActType.NOTIFICACION),
    (r"notificaciones",          ActType.NOTIFICACION),
    (r"recaudación ejecutiva",   ActType.NOTIFICACION),
    (r"propuesta de resolución", ActType.RESOLUCION),
    (r"bases",                   ActType.CONVOCATORIA),
    (r"convocatoria",            ActType.CONVOCATORIA),
    (r"nombramiento",            ActType.RESOLUCION),
    (r"delegación",              ActType.RESOLUCION),
    (r"emplazamiento",           ActType.NOTIFICACION),
    (r"citación",                ActType.NOTIFICACION),
    (r"diligencia",              ActType.NOTIFICACION),
    (r"cédula",                  ActType.NOTIFICACION),
    (r"requerimiento",           ActType.NOTIFICACION),
    (r"sala primera",            ActType.OTROS),
    (r"sala segunda",            ActType.OTROS),
    (r"__toponimo__",            ActType.OTROS),
    (r"__subasta_aeat__",        ActType.OTROS),
]


def inferir_act_type(description: str, bulletin: str) -> ActType:
    desc_clean = preprocess_description(description, bulletin)
    text = desc_clean.lower().strip()
    for pattern, act_type in _N1_MAP:
        if text.startswith(pattern):
            return act_type
    return ActType.OTROS

In [ ]:
df["act_type_n1"] = df.apply(
    lambda row: inferir_act_type(row["description"], row["bulletin"]), axis=1
)

dist = df["act_type_n1"].value_counts()
total = len(df)
print("Distribución N1 inferida:\n")
for val, count in dist.items():
    print(f"  {val:<25} {count:>6,}  ({count/total*100:.1f}%)")

otros = (df["act_type_n1"] == ActType.OTROS).sum()
print(f"\nCobertura N1: {(1 - otros/total)*100:.1f}%  ({otros:,} en OTROS)")

### Límites del clasificador N1 y trabajo futuro

El clasificador de primer token cubre **88.6%** del corpus con reglas deterministas.
El 11.4% restante cae en `OTROS` por tres motivos con soluciones conocidas:

| Grupo | Volumen | Motivo | Solución futura |
|---|---|---|---|
| Topónimos BOE / subastas AEAT | ~5.600 | Sin contenido textual real — irrecuperable sin PDF | Clase propia `NO_INFERIBLE` en v2 |
| BOCM/BOCA residual | ~300 | Variantes de cabecera no contempladas | Ampliar reglas de pre-procesamiento |
| RRHH sin tipo explícito | ~2.000 | `RELACIÓN`, `LISTA`, `OFERTA`... | LLM zero-shot viable en v2 |

---

## §3. Ground truth

El ground truth se construye en tres capas:

| Capa | Qué etiqueta | Cómo | Estado |
|------|-------------|------|--------|
| **1 — N1 determinista** | `act_type` | Reglas de primer token (§2) | ✅ Completo |
| **2 — Muestreo estratificado** | Selección de 100 registros | Keywords por procedimiento N2 | ✅ Completo |
| **3 — Anotación manual** | `is_relevant_gt`, `procedures_gt`, `technologies_gt` | Revisión humana | ✅ Completo (100 registros) |

El archivo `ground_truth_100_anotado.csv` contiene los 100 registros con etiquetas manuales validadas.

In [ ]:
import random
random.seed(42)

keywords = {
    "DIA":     ["declaración de impacto ambiental"],
    "AAP":     ["autorización administrativa previa"],
    "AAC":     ["autorización de construcción", "autorización administrativa de construcción"],
    "AAP_AAC": ["previa y de construcción"],
    "AAU":     ["autorización ambiental unificada"],
    "IIA":     ["informe de impacto ambiental"],
    "AAI":     ["autorización ambiental integrada"],
    "IAE":     ["ambiental estratégic"],
    "DUP":     ["utilidad pública"],
}
cuotas = {"DIA": 60, "AAP": 60, "AAC": 50, "AAP_AAC": 40,
          "AAU": 40, "IIA": 40, "AAI": 30, "IAE": 30, "DUP": 30}
N_NEGATIVOS = 120

sampled_ids = set()
frames = []

for grupo, kws in keywords.items():
    mask = df["description"].str.lower().str.contains("|".join(kws), na=False)
    mask = mask & ~df.index.isin(sampled_ids)
    pool = df[mask]
    n = min(cuotas[grupo], len(pool))
    sample = pool.sample(n, random_state=42).copy()
    sample["grupo_muestreo"] = grupo
    sampled_ids.update(sample.index.tolist())
    frames.append(sample)
    print(f"  {grupo:<10} pool={len(pool):>5,}  sampled={n}")

all_kws = [kw for kws in keywords.values() for kw in kws]
mask_neg = ~df["description"].str.lower().str.contains("|".join(all_kws), na=False)
mask_neg = mask_neg & ~df.index.isin(sampled_ids)
negativos = df[mask_neg].sample(N_NEGATIVOS, random_state=42).copy()
negativos["grupo_muestreo"] = "NEGATIVO"
frames.append(negativos)

df_gt_full = pd.concat(frames, ignore_index=True)
df_gt_full["id"] = range(len(df_gt_full))
print(f"\nTotal muestreado: {len(df_gt_full)} registros")

### Dataset anotado

Los 100 registros del ground truth han sido anotados manualmente. Durante el proceso se identificaron casos especiales documentados en `decisiones_implementacion.md` (DEC-012 a DEC-019):

- **Denegaciones**: heredan el tipo de procedimiento del acto denegado
- **Modificaciones**: heredan los procedimientos del acto modificado
- **Falsos positivos**: RRHH con vocabulario ambiental, concesiones de dominio público no energéticas
- **Tecnologías emergentes v2**: `industria_ippc`, `infraestructura_hidrica`, `ordenacion_territorial`, `turismo_edificacion`

In [ ]:
# Cargar ground truth anotado manualmente
PATH_GT_ANOTADO = "../data/ground_truth/ground_truth_100_anotado.csv"
df_anotado = pd.read_csv(PATH_GT_ANOTADO)

print(f"Ground truth: {len(df_anotado)} registros")
print(f"Relevantes: {df_anotado['is_relevant_gt'].sum()} | No relevantes: {(~df_anotado['is_relevant_gt'].astype(bool)).sum()}")
print()
print("Distribución N2:")
print(df_anotado["procedures_gt"].value_counts().to_string())

---

## §4. Agente base

El agente Pydantic AI recibe una descripción de boletín y devuelve un `ClassifierOutput` validado. Se compara en dos configuraciones:

- **Baseline**: input = `description` + `bulletin`. El LLM infiere `act_type` desde el texto.
- **+N1**: input = `description` + `bulletin` + `act_type` pre-computado. El LLM lo usa como contexto.

In [ ]:
SYSTEM_PROMPT = """
Eres un experto en clasificación de publicaciones de boletines oficiales españoles.
Tu tarea es analizar la descripción de una publicación y asignarle etiquetas según la taxonomía definida.

## Dominio
Las publicaciones relevantes pertenecen al universo de autorizaciones ambiental-energéticas (~3.7% del corpus).
El resto (RRHH, contratos, subvenciones, urbanismo...) son is_relevant=False.

## Procedimientos N2
| Etiqueta | Descripción |
|----------|-------------|
| DIA | Declaración de Impacto Ambiental — resolución que formula o aprueba el impacto ambiental |
| AAP | Autorización Administrativa Previa — valida el anteproyecto |
| AAC | Autorización Administrativa de Construcción — permiso definitivo de obras |
| AAU | Autorización Ambiental Unificada — equivalente regional a DIA en BOJA/DOE/BON |
| IIA | Informe de Impacto Ambiental — evaluación simplificada, distinta de DIA |
| AAI | Autorización Ambiental Integrada — permiso IPPC/IED, distinta de DIA |
| IAE | Informe/Declaración Ambiental Estratégico — aplica a planes y programas |
| DUP | Declaración de Utilidad Pública — reconoce interés general, habilita expropiación |

## Tecnologías N3
fotovoltaica · eólica · almacenamiento · hibridación · hidroeléctrica ·
biogás_biometano · biomasa · hidrógeno · línea_eléctrica · gas_natural · petróleo

## Reglas críticas
1. is_relevant=True SOLO si identificas al menos un procedimiento N2
2. AAU ≠ DIA — son procedimientos distintos aunque equivalentes funcionalmente
3. "autorización administrativa previa y de construcción" → [AAP, AAC] (no solo AAP)
4. "aprobación del proyecto de ejecución" junto a AAP → añadir AAC
5. IIA ≠ DIA — el informe de impacto ambiental es evaluación simplificada
6. AAI ≠ DIA — solo es DIA si menciona explícitamente "declaración de impacto ambiental"
7. IAE aplica a planes y programas, no a proyectos individuales
8. DUP puede acompañar a AAP/AAC pero no es AAP ni AAC por sí sola
9. Las denegaciones y desistimientos heredan el tipo del procedimiento denegado
10. Las modificaciones heredan los procedimientos del acto modificado
11. reasoning debe citar el fragmento exacto del texto que dispara cada etiqueta
""".strip()

In [ ]:
agent = Agent(
    model,
    output_type=ClassifierOutput,
    system_prompt=SYSTEM_PROMPT,
)

print("✓ Agente construido")

In [ ]:
async def clasificar(description: str, bulletin: str, use_n1_context: bool = False) -> ClassifierOutput:
    n0 = get_ambito(bulletin)
    user_msg = f"Boletín: {bulletin.upper()} (ámbito: {n0})\n\nDescripción: {description}"
    if use_n1_context:
        act_type_pre = inferir_act_type(description, bulletin)
        user_msg += f"\n\nTipo de acto pre-clasificado (N1): {act_type_pre.value}"
    result = await agent.run(user_msg)
    return result.output

In [ ]:
# Casos cualitativos — 5 ejemplos de validación
casos = [
    ("Resolución de 12 de marzo de 2025, de la Dirección General de Calidad y Evaluación "
     "Ambiental, por la que se formula la declaración de impacto ambiental del proyecto "
     "Planta Solar Fotovoltaica Los Llanos, en la provincia de Cáceres.", "doe"),
    ("Resolución de 5 de febrero de 2025, de la Dirección General de Política Energética, "
     "por la que se otorga autorización administrativa previa y de construcción para el "
     "Parque Eólico Sierra Norte, de 48 MW, en Salamanca.", "boe"),
    ("Resolución de 18 de enero de 2025, de la Delegación Territorial de Medio Ambiente, "
     "por la que se otorga autorización ambiental unificada para la planta de biogás "
     "Valdecorneja, en Ávila.", "boja"),
    ("Resolución de 3 de marzo de 2025, de la Universidad de Salamanca, por la que se "
     "convoca concurso-oposición para cubrir plazas de profesor ayudante doctor.", "bocyl"),
    ("Resolución de 21 de febrero de 2025, de la Dirección General de Medio Natural, "
     "por la que se formula el informe de impacto ambiental del proyecto de línea "
     "eléctrica subterránea de 132 kV en Zaragoza.", "boa"),
]

EXPECTED = [
    {"procedures": {"DIA"}, "technologies": {"fotovoltaica"}, "relevant": True},
    {"procedures": {"AAP","AAC"}, "technologies": {"eólica"}, "relevant": True},
    {"procedures": {"AAU"}, "technologies": {"biogás_biometano"}, "relevant": True},
    {"procedures": set(), "technologies": set(), "relevant": False},
    {"procedures": {"IIA"}, "technologies": {"línea_eléctrica"}, "relevant": True},
]

print("Validación cualitativa — 5 casos\n")
aciertos = 0
for i, (desc, bul) in enumerate(casos, 1):
    r = await clasificar(desc, bul)
    pred_proc = set(p.value for p in r.procedures)
    pred_tech = set(t.value for t in r.technologies)
    exp = EXPECTED[i-1]
    ok = (r.is_relevant == exp["relevant"] and pred_proc == exp["procedures"])
    aciertos += ok
    mark = "✅" if ok else "❌"
    print(f"{mark} Caso {i} | is_relevant={r.is_relevant} | procedures={pred_proc} | technologies={pred_tech}")
    if not ok:
        print(f"   Esperado: relevant={exp['relevant']} procedures={exp['procedures']}")
print(f"\nResultado: {aciertos}/5 correctos")

---

## §5. Experimento 1 — Baseline

Configuración zero-shot sin contexto N1. El LLM recibe solo `description` + `bulletin` y debe inferir todos los campos por sí solo.

**Modelo**: Qwen 3.5 9B (thinking desactivado)
**Registros**: 100 (muestra estratificada del ground truth)
**Concurrencia**: 1 (DEC-009: modelos locales procesan secuencialmente)

In [ ]:
from tqdm.asyncio import tqdm_asyncio


async def clasificar_async(
    description: str,
    bulletin: str,
    use_n1_context: bool = False,
) -> dict:
    n0 = get_ambito(bulletin)
    user_msg = f"Boletín: {bulletin.upper()} (ámbito: {n0})\n\nDescripción: {description}"
    if use_n1_context:
        act_type_pre = inferir_act_type(description, bulletin)
        user_msg += f"\n\nTipo de acto pre-clasificado (N1): {act_type_pre.value}"
    try:
        result = await agent.run(user_msg)
        output = result.output
        return {
            "is_relevant_pred": output.is_relevant,
            "act_type_pred": output.act_type.value,
            "procedures_pred": json.dumps([p.value for p in output.procedures], ensure_ascii=False),
            "technologies_pred": json.dumps([t.value for t in output.technologies], ensure_ascii=False),
            "confidence": output.confidence,
            "reasoning": output.reasoning,
        }
    except Exception as e:
        return {
            "is_relevant_pred": None, "act_type_pred": None,
            "procedures_pred": "[]", "technologies_pred": "[]",
            "confidence": None, "reasoning": f"ERROR: {str(e)[:100]}",
        }


async def run_experiment(
    df_input: pd.DataFrame,
    use_n1_context: bool = False,
    concurrency: int = 1,
    output_path: str = "../results/experiment.csv",
) -> pd.DataFrame:
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    semaphore = asyncio.Semaphore(concurrency)

    async def process_row(row):
        async with semaphore:
            pred = await clasificar_async(row["description"], row["bulletin"], use_n1_context)
            return {**row.to_dict(), **pred}

    tasks = [process_row(row) for _, row in df_input.iterrows()]
    results = await tqdm_asyncio.gather(*tasks, desc="Clasificando")
    df_results = pd.DataFrame(results)
    df_results.to_csv(output_path, index=False)
    errores = df_results["reasoning"].astype(str).str.startswith("ERROR").sum()
    print(f"\n✓ Guardado en {output_path} | Errores: {errores}/{len(df_results)}")
    return df_results

In [ ]:
# ── Ejecutar Experimento 1 — Baseline ────────────────────────────────────────
# Requiere LM Studio activo con el modelo cargado

df_exp1 = await run_experiment(
    df_anotado,
    use_n1_context=False,
    concurrency=1,
    output_path="../results/exp1_baseline_qwen9b.csv",
)

print(df_exp1[["grupo_muestreo", "procedures_pred", "confidence"]].head(10))

---

## §6. Análisis de errores — Baseline

Evaluamos las predicciones del Experimento 1 contra las etiquetas manuales del ground truth. El objetivo es identificar patrones de error sistemáticos que guíen la mejora del prompt en §7.

In [ ]:
def parse_labels(value) -> set:
    if pd.isna(value) or str(value).strip() in ("", "nan"): return set()
    v = str(value).strip()
    if v.startswith("["):
        try: return set(json.loads(v))
        except: pass
    return set(x.strip() for x in v.split(",") if x.strip())


def compute_metrics(df_eval):
    """Calcula métricas completas sobre el DataFrame mergeado."""
    y_true = df_eval["is_relevant_gt"].astype(bool)
    y_pred = df_eval["is_relevant_pred"].fillna(False).astype(bool)

    # is_relevant
    tp=((y_true)&(y_pred)).sum(); fp=((~y_true)&(y_pred)).sum()
    fn=((y_true)&(~y_pred)).sum(); tn=((~y_true)&(~y_pred)).sum()
    p=tp/(tp+fp) if tp+fp>0 else 0; r=tp/(tp+fn) if tp+fn>0 else 0
    f1_rel=2*p*r/(p+r) if p+r>0 else 0

    print(f"── is_relevant  P={p:.3f}  R={r:.3f}  F1={f1_rel:.3f}  TP={tp} FP={fp} FN={fn} TN={tn}\n")

    # N2
    N2 = ["DIA","AAP","AAC","AAU","IIA","AAI","IAE","DUP"]
    print(f"{'Label':<8} {'P':>6} {'R':>6} {'F1':>6} {'Sup':>5} {'TP':>4} {'FP':>4} {'FN':>4}")
    print("─" * 55)
    macro = 0
    for label in N2:
        yt = df_eval.apply(lambda r: label in parse_labels(r["procedures_gt"]), axis=1)
        yp = df_eval.apply(lambda r: label in parse_labels(r["procedures_pred"]), axis=1)
        tp2=(yt&yp).sum(); fp2=(~yt&yp).sum(); fn2=(yt&~yp).sum()
        p2=tp2/(tp2+fp2) if tp2+fp2>0 else 0
        r2=tp2/(tp2+fn2) if tp2+fn2>0 else 0
        f12=2*p2*r2/(p2+r2) if p2+r2>0 else 0
        sup=yt.sum(); macro+=f12
        print(f"{label:<8} {p2:>6.3f} {r2:>6.3f} {f12:>6.3f} {sup:>5} {tp2:>4} {fp2:>4} {fn2:>4}")
    print("─" * 55)
    print(f"{'Macro-F1':<8} {macro/len(N2):>6.3f}")

    # Exact match
    exact = df_eval.apply(
        lambda r: parse_labels(r["procedures_gt"]) == parse_labels(r["procedures_pred"]), axis=1
    )
    rel = y_true
    print(f"\nExact match (relevantes): {exact[rel].mean():.3f}  ({exact[rel].sum()}/{rel.sum()})")
    print(f"Exact match (todos):      {exact.mean():.3f}  ({exact.sum()}/{len(df_eval)})")

    # Confidence
    conf = df_eval["confidence"].dropna()
    print(f"\nConfianza: media={conf.mean():.3f}  min={conf.min():.3f}  max={conf.max():.3f}")

    return exact, rel

In [ ]:
# Cargar resultados del Experimento 1
df_exp1 = pd.read_csv("../results/exp1_baseline_qwen9b.csv")

# Merge con ground truth por descripción
df_eval1 = df_anotado[["id","is_relevant_gt","procedures_gt","technologies_gt","description"]].merge(
    df_exp1[["description","is_relevant_pred","act_type_pred","procedures_pred",
              "technologies_pred","confidence","reasoning"]],
    on="description", how="left"
)

print(f"Registros evaluados: {len(df_eval1)}")
print(f"Predicciones válidas: {df_eval1['is_relevant_pred'].notna().sum()}\n")

exact1, rel1 = compute_metrics(df_eval1)

In [ ]:
# Análisis detallado de errores N2
print("── Errores N2 en registros relevantes ──────────────────────────────\n")
errores1 = df_eval1[~exact1 & rel1]
print(f"Total errores: {len(errores1)}\n")

for _, row in errores1.iterrows():
    gt = parse_labels(row["procedures_gt"])
    pred = parse_labels(row["procedures_pred"])
    missing = sorted(gt - pred)
    extra = sorted(pred - gt)
    print(f"ID {row['id']} | {str(row.get('bulletin','')[:10])}")
    print(f"  GT  : {sorted(gt)}")
    print(f"  PRED: {sorted(pred)}")
    if missing: print(f"  Falta : {missing}")
    if extra:   print(f"  Sobra : {extra}")
    print(f"  Desc: {str(row['description'])[:100]}...")
    print(f"  Razón: {str(row['reasoning'])[:120]}")
    print()

---

## §7. Prompt v2 — Mejora basada en errores

Basándonos en el análisis de errores de §6, mejoramos el system prompt incorporando las reglas críticas identificadas durante el etiquetado manual (DEC-013 a DEC-019).

**Cambios respecto al Prompt v1:**
- Reglas explícitas para denegaciones y desistimientos (DEC-013)
- Reglas para modificaciones de actos previos (DEC-013)
- Distinción explícita IIA vs DIA con ejemplos
- Distinción explícita AAI vs DIA
- Manejo de "evaluación de impacto ambiental" en solicitudes conjuntas (PENDIENTE-001)
- Ejemplos negativos: RRHH con vocabulario ambiental, concesiones no energéticas (DEC-018, DEC-019)

In [ ]:
SYSTEM_PROMPT_V2 = """
Eres un experto en clasificación de publicaciones de boletines oficiales españoles.
Tu tarea es analizar la descripción de una publicación y asignarle etiquetas según la taxonomía definida.

## Dominio
Las publicaciones relevantes pertenecen al universo de autorizaciones ambiental-energéticas (~3.7% del corpus).
El resto (RRHH, contratos, subvenciones, urbanismo, telecomunicaciones...) son is_relevant=False.

## Procedimientos N2
| Etiqueta | Descripción |
|----------|-------------|
| DIA | Declaración de Impacto Ambiental — resolución que formula o aprueba el impacto ambiental |
| AAP | Autorización Administrativa Previa — valida el anteproyecto |
| AAC | Autorización Administrativa de Construcción — permiso definitivo de obras |
| AAU | Autorización Ambiental Unificada — equivalente regional a DIA en BOJA/DOE/BON. DISTINTA de DIA |
| IIA | Informe de Impacto Ambiental — evaluación SIMPLIFICADA. DISTINTA de DIA |
| AAI | Autorización Ambiental Integrada — permiso IPPC/IED. DISTINTA de DIA salvo mención explícita |
| IAE | Informe/Declaración Ambiental Estratégico — aplica SOLO a planes y programas, no a proyectos |
| DUP | Declaración de Utilidad Pública — reconoce interés general, habilita expropiación |

## Tecnologías N3
fotovoltaica · eólica · almacenamiento · hibridación · hidroeléctrica ·
biogás_biometano · biomasa · hidrógeno · línea_eléctrica · gas_natural · petróleo

## Reglas críticas — SEGUIR ESTRICTAMENTE

### Combinaciones frecuentes
- "autorización administrativa previa y de construcción" → [AAP, AAC] (ambas, nunca solo AAP)
- "aprobación del proyecto de ejecución" junto a AAP → añadir AAC
- DUP puede aparecer junto a AAP+AAC en la misma resolución → [AAP, AAC, DUP]

### Denegaciones y desistimientos (DEC-013)
- Una resolución que DENIEGA o DESESTIMA una AAP → etiquetar igualmente como AAP
- Un desistimiento de AAP+AAC+DUP → etiquetar como [AAP, AAC, DUP]
- El tipo de procedimiento es INDEPENDIENTE del sentido de la resolución

### Modificaciones (DEC-013)
- "modificación de la AAP y AAC" → etiquetar como [AAP, AAC]
- "modificación de informe vinculante sobre AAU" → etiquetar como [AAU]
- Las modificaciones HEREDAN los procedimientos del acto modificado

### Confusiones críticas a evitar
- IIA ≠ DIA: "informe de impacto ambiental" es evaluación simplificada, NO es declaración
- AAI ≠ DIA: "autorización ambiental integrada" es permiso IPPC, NO es DIA
- AAU ≠ DIA: aunque funcionalmente equivalentes, son etiquetas DISTINTAS
- IAE aplica a PLANES (urbanísticos, sectoriales), no a proyectos de instalaciones

### Falsos positivos a evitar (is_relevant=False)
- Procesos selectivos de RRHH aunque mencionen "medioambiental" en el título del puesto
- Concesiones de dominio público para telecomunicaciones (fibra óptica, antenas 5G)
- Programas de inspección ambiental (no son autorizaciones individuales)
- Padrones fiscales, convenios de transporte, plantillas orgánicas

### reasoning
Citar el fragmento EXACTO del texto que dispara cada etiqueta. Máximo 2 frases.
""".strip()

---

## §8. Experimento 2 — Prompt v2

Mismo modelo (Qwen 3.5 9B), mismos 100 registros, pero con el prompt mejorado. Comparamos con Experimento 1 para cuantificar la ganancia del prompt engineering.

In [ ]:
agent_v2 = Agent(
    model,
    output_type=ClassifierOutput,
    system_prompt=SYSTEM_PROMPT_V2,
)


async def clasificar_async_v2(description: str, bulletin: str, use_n1_context: bool = False) -> dict:
    n0 = get_ambito(bulletin)
    user_msg = f"Boletín: {bulletin.upper()} (ámbito: {n0})\n\nDescripción: {description}"
    if use_n1_context:
        act_type_pre = inferir_act_type(description, bulletin)
        user_msg += f"\n\nTipo de acto pre-clasificado (N1): {act_type_pre.value}"
    try:
        result = await agent_v2.run(user_msg)
        output = result.output
        return {
            "is_relevant_pred": output.is_relevant,
            "act_type_pred": output.act_type.value,
            "procedures_pred": json.dumps([p.value for p in output.procedures], ensure_ascii=False),
            "technologies_pred": json.dumps([t.value for t in output.technologies], ensure_ascii=False),
            "confidence": output.confidence,
            "reasoning": output.reasoning,
        }
    except Exception as e:
        return {
            "is_relevant_pred": None, "act_type_pred": None,
            "procedures_pred": "[]", "technologies_pred": "[]",
            "confidence": None, "reasoning": f"ERROR: {str(e)[:100]}",
        }


async def run_experiment_v2(df_input, use_n1_context=False, concurrency=1, output_path="../results/exp.csv"):
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    semaphore = asyncio.Semaphore(concurrency)

    async def process_row(row):
        async with semaphore:
            pred = await clasificar_async_v2(row["description"], row["bulletin"], use_n1_context)
            return {**row.to_dict(), **pred}

    tasks = [process_row(row) for _, row in df_input.iterrows()]
    results = await tqdm_asyncio.gather(*tasks, desc="Clasificando v2")
    df_results = pd.DataFrame(results)
    df_results.to_csv(output_path, index=False)
    errores = df_results["reasoning"].astype(str).str.startswith("ERROR").sum()
    print(f"\n✓ Guardado en {output_path} | Errores: {errores}/{len(df_results)}")
    return df_results


df_exp2 = await run_experiment_v2(
    df_anotado,
    use_n1_context=False,
    concurrency=1,
    output_path="../results/exp2_promptv2_qwen9b.csv",
)

In [ ]:
df_exp2 = pd.read_csv("../results/exp2_promptv2_qwen9b.csv")
df_eval2 = df_anotado[["id","is_relevant_gt","procedures_gt","technologies_gt","description"]].merge(
    df_exp2[["description","is_relevant_pred","procedures_pred","technologies_pred","confidence","reasoning"]],
    on="description", how="left"
)

print("── Experimento 2 — Prompt v2 ──────────────────────────")
exact2, rel2 = compute_metrics(df_eval2)

---

## §9. Experimento 3 — Ablación +N1

Añadimos el `act_type` pre-computado por reglas como contexto al prompt. Pregunta de investigación: **¿cuánto aporta saber la forma jurídica del documento para clasificar el procedimiento N2?**

Usamos Prompt v2 + contexto N1.

In [ ]:
df_exp3 = await run_experiment_v2(
    df_anotado,
    use_n1_context=True,  # ← única diferencia
    concurrency=1,
    output_path="../results/exp3_promptv2_n1_qwen9b.csv",
)

In [ ]:
df_exp3 = pd.read_csv("../results/exp3_promptv2_n1_qwen9b.csv")
df_eval3 = df_anotado[["id","is_relevant_gt","procedures_gt","technologies_gt","description"]].merge(
    df_exp3[["description","is_relevant_pred","procedures_pred","technologies_pred","confidence","reasoning"]],
    on="description", how="left"
)

print("── Experimento 3 — Prompt v2 + N1 ────────────────────")
exact3, rel3 = compute_metrics(df_eval3)

---

## §10. Experimento 4 — Few-shot

Añadimos ejemplos reales al prompt (one o two-shot por categoría difícil). Pregunta de investigación: **¿mejora la clasificación de los casos borde cuando el modelo tiene ejemplos concretos?**

Los ejemplos se seleccionan de los errores identificados en §6.

> ⚙️ **Pendiente**: implementar tras analizar los errores del Experimento 1.

In [ ]:
# TODO: implementar tras análisis de errores §6
# Los ejemplos few-shot se seleccionarán de los casos de error identificados
# con énfasis en: IIA vs DIA, AAI vs DIA, denegaciones, modificaciones
print("Pendiente de implementar tras análisis de errores §6")

---

## §11. Comparativa de modelos

Repetimos el mejor experimento (Prompt v2 + configuración óptima) con Gemma 4 4B.
Pregunta de investigación: **¿cuánto se pierde en F1 al usar un modelo 4B vs 9B?**

Para cambiar al Gemma 4B: en §0, cambiar `LM_STUDIO_MODEL = "gemma-4-4b-it"` y reiniciar el kernel.

> ⚙️ **Pendiente**: ejecutar cuando Gemma 4B esté descargado en LM Studio.

In [ ]:
# TODO: ejecutar con Gemma 4B
# Cambiar LM_STUDIO_MODEL en §0 y re-ejecutar el mejor experimento
# Guardar en ../results/exp_best_gemma4b.csv
print("Pendiente: descargar Gemma 4B en LM Studio y ejecutar")

---

## §12. Tabla resumen — Comparativa de experimentos

In [ ]:
# Resumen automático de todos los experimentos ejecutados
experimentos = [
    ("Exp 1 — Baseline", "../results/exp1_baseline_qwen9b.csv"),
    ("Exp 2 — Prompt v2", "../results/exp2_promptv2_qwen9b.csv"),
    ("Exp 3 — Prompt v2 + N1", "../results/exp3_promptv2_n1_qwen9b.csv"),
]

N2 = ["DIA","AAP","AAC","AAU","IIA","AAI","IAE","DUP"]

print(f"{'Experimento':<25} {'is_rel F1':>10} {'Macro-F1':>10} {'Exact(rel)':>12} {'Errores':>8}")
print("─" * 70)

for nombre, path in experimentos:
    try:
        df_r = pd.read_csv(path)
        df_e = df_anotado[["id","is_relevant_gt","procedures_gt","description"]].merge(
            df_r[["description","is_relevant_pred","procedures_pred","confidence","reasoning"]],
            on="description", how="left"
        )
        y_true = df_e["is_relevant_gt"].astype(bool)
        y_pred = df_e["is_relevant_pred"].fillna(False).astype(bool)
        tp=((y_true)&(y_pred)).sum(); fp=((~y_true)&(y_pred)).sum()
        fn=((y_true)&(~y_pred)).sum()
        p=tp/(tp+fp) if tp+fp>0 else 0; r=tp/(tp+fn) if tp+fn>0 else 0
        f1_rel=2*p*r/(p+r) if p+r>0 else 0

        macro=0
        for label in N2:
            yt=df_e.apply(lambda r: label in parse_labels(r["procedures_gt"]), axis=1)
            yp=df_e.apply(lambda r: label in parse_labels(r["procedures_pred"]), axis=1)
            tp2=(yt&yp).sum(); fp2=(~yt&yp).sum(); fn2=(yt&~yp).sum()
            p2=tp2/(tp2+fp2) if tp2+fp2>0 else 0; r2=tp2/(tp2+fn2) if tp2+fn2>0 else 0
            macro+=2*p2*r2/(p2+r2) if p2+r2>0 else 0

        exact=df_e.apply(lambda r: parse_labels(r["procedures_gt"])==parse_labels(r["procedures_pred"]), axis=1)
        errores=df_e["reasoning"].astype(str).str.startswith("ERROR").sum()
        print(f"{nombre:<25} {f1_rel:>10.3f} {macro/len(N2):>10.3f} {exact[y_true].mean():>12.3f} {errores:>8}")
    except FileNotFoundError:
        print(f"{nombre:<25} {'(pendiente)':>10}")

---

## §13. Calibración de confianza

Analizamos si el campo `confidence` es un predictor real de calidad. **Hipótesis**: los registros con `confidence < 0.8` deberían tener peor F1 que los de `confidence > 0.95`.

Si el modelo es sobreconfiante (todos los valores entre 0.95-1.0), el campo `confidence` no sirve como filtro práctico — esto es un hallazgo relevante para la memoria.

In [ ]:
try:
    df_cal = pd.read_csv("../results/exp2_promptv2_qwen9b.csv")
    df_c = df_anotado[["id","is_relevant_gt","procedures_gt","description"]].merge(
        df_cal[["description","is_relevant_pred","procedures_pred","confidence"]],
        on="description", how="left"
    )
    df_c = df_c[df_c["confidence"].notna()]

    print(f"Distribución de confidence:")
    bins = [0, 0.7, 0.8, 0.9, 0.95, 1.01]
    labels_b = ["<0.7","0.7-0.8","0.8-0.9","0.9-0.95",">=0.95"]
    df_c["conf_bin"] = pd.cut(df_c["confidence"], bins=bins, labels=labels_b, right=False)
    print(df_c["conf_bin"].value_counts().sort_index().to_string())
    print()

    print(f"Exact match por rango de confianza:")
    df_c["exact"] = df_c.apply(
        lambda r: parse_labels(r["procedures_gt"]) == parse_labels(r["procedures_pred"]), axis=1
    )
    for bin_label in labels_b:
        subset = df_c[df_c["conf_bin"] == bin_label]
        if len(subset) > 0:
            print(f"  {bin_label:<10}: {subset['exact'].mean():.3f}  (n={len(subset)})")

    mean_conf = df_c["confidence"].mean()
    overconfident = (df_c["confidence"] >= 0.95).mean()
    print(f"\nMedia confianza: {mean_conf:.3f}")
    print(f"Registros con conf >= 0.95: {overconfident:.1%}")
    if overconfident > 0.8:
        print("⚠️  Modelo sobreconfiante — confidence no sirve como filtro práctico")
except FileNotFoundError:
    print("Pendiente: ejecutar Experimento 2 primero")

---

## §14. Conclusiones y trabajo futuro

### Hallazgos principales

> *Completar tras ejecutar todos los experimentos*

### Limitaciones

1. **Ground truth Opción A**: el dataset de 100 registros fue anotado manualmente sin revisión cruzada. La Opción B (anotación rigurosa con múltiples anotadores) queda como trabajo futuro.
2. **Modelo local limitado**: Qwen 3.5 9B con 16GB RAM es el modelo más grande ejecutable en el hardware disponible. Los experimentos con modelos frontera (Gemini 2.5 Pro) se limitan a mini-muestras por rate limits.
3. **Sobreconfianza**: el campo `confidence` no es un predictor fiable de calidad en modelos locales — todos los valores tienden a 0.95-1.0.

### Trabajo futuro — v2

| Tarea | Descripción |
|-------|-------------|
| Ampliar corpus | Extender a las 18 familias y 92 procedimientos del N2 completo |
| Ground truth riguroso | Opción B: anotación manual con revisión cruzada |
| Nuevas tecnologías N3 | `industria_ippc`, `infraestructura_hidrica`, `ordenacion_territorial`, `turismo_edificacion` |
| N1 para RRHH | LLM zero-shot para registros sin tipo de acto explícito |
| Modelos frontera | Evaluación completa con Gemini 2.5 Pro / GPT-4o |